# Autoportrait 2 — description des autoportraits avec Qwen3-32B (consigne A)

Ce carnet est la **deuxième phase** de l'étude. La première phase,
[autoportrait_qwen3_32b.ipynb](autoportrait_qwen3_32b.ipynb), classe chaque chanson du corpus
d'étude stratifié en `autoportrait`, `pas_autoportrait` ou `je_ne_sais_pas`. Ici, on ne garde
que le **sous-ensemble AUTOPORTRAIT** et on **décrit** chaque autoportrait selon
[consigne_A_description.md](consigne_A_description.md). La réponse doit respecter le schéma
[schema_A_description.json](schema_A_description.json).

| Champ | Valeurs | Question |
|---|---|---|
| `justification` | 1–2 phrases, sans citation | résumé du portrait |
| `registre` | `egotrip`, `introspection`, `manifeste`, `mixte` | sur quel ton se peint-il ? |
| `aspects` | 1 à 4 parmi 12 aspects, **du plus au moins important** | de quoi son portrait est-il fait ? |
| `temporalite` | `present`, `passe`, `trajectoire`, `avenir` | à quel moment de sa vie ? |
| `demarcation` | 0 à 7 figures (liste vide possible) | contre qui se définit-il ? |
| `regard_des_autres` | `oui` / `non` | rapporte-t-il le regard des autres ? |

## Plan

0. **Configuration** : paramètres à régler (en tête de la première cellule) et chemins.
1. **Chargement du sous-ensemble AUTOPORTRAIT** : lecture des résultats de la phase 1
   (`autoportrait_qwen_annotation_{n_source}.csv`), filtrage `etiquette == "autoportrait"`
   et `statut == "ok"`, puis récupération des paroles dans `Rap/corpus.csv` par `source_row_index`.
2. **Échantillonnage : test ou sous-ensemble complet** : `taille_echantillon` entier = test
   tiré au hasard (graine 42) parmi les autoportraits ; `None` = tous les autoportraits.
3. **Annotation** : client vLLM (3.1), pipeline avec reprise (3.2), lancement (3.3).
4. **Synthèse** : distributions de chaque champ et croisement registre × période.

## Paramètres (première cellule)

- `n_source` : taille du lot de phase 1 à exploiter. `12938` = corpus d'étude stratifié complet,
  disponible une fois la campagne de phase 1 terminée. Pour un essai avant la fin de cette campagne,
  on peut prendre un lot déjà terminé (par ex. `1024`).
- `taille_echantillon` : `10` pour les petites mises au point, `None` pour tout le sous-ensemble.
- `workers_par_gpu` : requêtes simultanées par serveur vLLM (≤ `--max-num-seqs`, 12 actuellement).
- `taille_lot` : nombre de chansons soumises à la fois (mémoire), sans effet sur le total.

## Inférence

Mêmes serveurs que la phase 1 : `Qwen/Qwen3-32B` en **mode raisonnement**
(`enable_thinking=True`, `--reasoning-parser qwen3`), GPU 1 sur le port 8016 et GPU 2 sur le port 8017
(`QWEN_BASE_URLS` pour changer). Le carnet ne démarre aucun serveur : **vérifier les GPU avec
`nvidia-smi` avant tout lancement**. Si la campagne de phase 1 tourne encore, les deux phases se
partagent les serveurs et chacune ralentit.

Le schéma A est imposé à vLLM par `response_format` (sortie JSON contrainte), puis **revalidé côté
client** : clés exactes, valeurs dans les listes autorisées, 1 à 4 aspects distincts, démarcations
distinctes, justification non vide, raisonnement présent. Une réponse invalide est retentée
(`max_essais`), puis journalisée en erreur.

## Sorties (`data/results/`, `data/performance/`)

- `autoportrait2_qwen_description_{n}.csv` : une ligne par chanson ; listes jointes par `|`,
  `aspect_principal` (premier aspect), indicatrices 0/1 `aspect_*` et `demarcation_*`, raisonnement.
- `autoportrait2_qwen_description_{n}.json` : mêmes données, listes conservées.
- `autoportrait2_qwen_description_{n}_erreurs.csv` : échecs techniques (relancer pour les retenter).
- `autoportrait2_qwen_description_{n}_distributions.csv` : synthèse (étape 4).
- `autoportrait2_qwen32_reason_{perimetre}_n{n}_{empreinte}.jsonl` : journal de reprise.
- `data/performance/perfGPU_autoportrait2.csv` : performances cumulées d'essai en essai,
  colonnes `date,modele,batch,concurrent,n,chansons/s` (fichier distinct de la phase 1).

`{n}` est le nombre de chansons traitées dans le lot.

In [ ]:
# ====================== PARAMÈTRES À RÉGLER ======================
# --- 1. Source : résultats de la phase 1 (autoportrait_qwen3_32b.ipynb) ---
# Lit data/results/autoportrait_qwen_annotation_{n_source}.csv ; 12938 = corpus d'étude stratifié complet.
n_source = 12938
# --- 2. Échantillonnage parmi les autoportraits ---
# None = tous les autoportraits du lot source ; un entier (ex. 10) = test tiré au hasard parmi eux.
taille_echantillon = None
# --- 3. Annotation ---
# Requêtes envoyées en parallèle à CHAQUE serveur vLLM (total = workers_par_gpu × nombre de serveurs).
# Garder workers_par_gpu ≤ --max-num-seqs des serveurs (12 actuellement).
workers_par_gpu = 12
# Nombre maximal de chansons soumises à la fois (mémoire) ; ne change pas le total.
taille_lot = 128
# ==================================================================

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display
from queue import Queue
import fcntl
import hashlib
import importlib.util
import json
import os
import time
import urllib.error
import urllib.request

import pandas as pd
from tqdm.auto import tqdm

# Accepte un lancement depuis la racine du dépôt, Rap ou Autoportrait.
candidats = [Path.cwd(), Path.cwd() / 'Autoportrait', Path.cwd() / 'Rap' / 'Autoportrait']
dossiers = [p.resolve() for p in candidats
            if (p / 'consigne_A_description.md').is_file() and (p / 'annoter.py').is_file()]
if len(dossiers) != 1:
    raise RuntimeError('Lancer le carnet depuis la racine du dépôt, Rap ou Rap/Autoportrait.')
dossier = dossiers[0]
corpus_path = dossier.parent / 'corpus.csv'
consigne_path = dossier / 'consigne_A_description.md'
schema_path = dossier / 'schema_A_description.json'
resultats_dir = dossier / 'data' / 'results'
performance_dir = dossier / 'data' / 'performance'
source_path = resultats_dir / f'autoportrait_qwen_annotation_{n_source}.csv'
cumul_path = performance_dir / 'perfGPU_autoportrait2.csv'

urls_configurees = os.environ.get('QWEN_BASE_URLS', os.environ.get(
    'QWEN_BASE_URL', 'http://127.0.0.1:8016/v1,http://127.0.0.1:8017/v1'))
base_urls = [url.strip().rstrip('/') for url in urls_configurees.split(',')]
if any(not url.startswith(('http://', 'https://')) for url in base_urls) or len(set(base_urls)) != len(base_urls):
    raise ValueError('Configurer des URL HTTP(S) distinctes dans QWEN_BASE_URLS.')
modele = 'Qwen/Qwen3-32B'
cle_api = os.environ.get('QWEN_API_KEY', 'EMPTY')
graine = 42
concurrence_par_serveur = workers_par_gpu
concurrence = concurrence_par_serveur * len(base_urls)
serveurs_disponibles = Queue()
for _ in range(concurrence_par_serveur):
    for url in base_urls:
        serveurs_disponibles.put(url)
timeout_secondes = 600
max_tokens = 8192
enable_thinking = True
max_essais = 2

if type(n_source) is not int or n_source <= 0:
    raise ValueError('n_source doit être un entier positif (taille du lot de phase 1).')
if taille_echantillon is not None and (type(taille_echantillon) is not int or taille_echantillon <= 0):
    raise ValueError('taille_echantillon doit être un entier positif, ou None pour tous les autoportraits.')
if type(workers_par_gpu) is not int or type(taille_lot) is not int:
    raise ValueError('workers_par_gpu et taille_lot doivent être des entiers positifs.')
if min(taille_lot, concurrence, timeout_secondes, max_tokens, max_essais) <= 0:
    raise ValueError('Les tailles, délais et nombres de tentatives doivent être positifs.')
spec = importlib.util.spec_from_file_location('autoportrait_source', dossier / 'annoter.py')
if spec is None or spec.loader is None:
    raise ImportError('Impossible de charger annoter.py.')
source = importlib.util.module_from_spec(spec)
spec.loader.exec_module(source)
consigne = consigne_path.read_text(encoding='utf-8').strip()
if not consigne:
    raise ValueError('La consigne A est vide.')
schema = json.loads(schema_path.read_text(encoding='utf-8'))
champs = list(schema['required'])
proprietes = schema['properties']
if set(champs) != set(proprietes) or champs[0] != 'justification':
    raise ValueError('Schéma A inattendu : champs requis et propriétés doivent coïncider.')
champs_simples = [c for c in champs if proprietes[c]['type'] == 'string' and 'enum' in proprietes[c]]
champs_listes = [c for c in champs if proprietes[c]['type'] == 'array']
modalites = {c: proprietes[c]['enum'] for c in champs_simples}
modalites |= {c: proprietes[c]['items']['enum'] for c in champs_listes}

def sans_unique_items(noeud):
    # xgrammar (vLLM) refuse « uniqueItems » ; l'unicité est vérifiée côté client par valider_description.
    if isinstance(noeud, dict):
        return {k: sans_unique_items(v) for k, v in noeud.items() if k != 'uniqueItems'}
    return [sans_unique_items(v) for v in noeud] if isinstance(noeud, list) else noeud

schema_serveur = sans_unique_items(schema)
contrainte = {'response_format': {'type': 'json_schema',
              'json_schema': {'name': 'description_autoportrait', 'schema': schema_serveur, 'strict': True}}}
print('Dossier :', dossier)
print('Source phase 1 :', source_path.name)
print('Modèle :', modele, '| endpoints :', base_urls, '| raisonnement :', enable_thinking)
print('Workers :', concurrence, 'au total |', workers_par_gpu, 'par GPU/serveur')
print('Champs :', champs)

## 1. Chargement du sous-ensemble AUTOPORTRAIT

On lit les résultats de la phase 1 pour le lot `n_source`
(`data/results/autoportrait_qwen_annotation_{n_source}.csv`, produit par
[autoportrait_qwen3_32b.ipynb](autoportrait_qwen3_32b.ipynb)) et on garde les chansons
**étiquetées `autoportrait` avec `statut == "ok"`**. Les `pas_autoportrait`, `je_ne_sais_pas`
et éventuelles erreurs techniques de la phase 1 sont écartés.

Ce CSV ne contient pas les paroles : on les relit dans `Rap/corpus.csv` par `source_row_index`
(position de la ligne de données, à partir de 0) avec `lire_corpus`, puis on applique le même
nettoyage qu'en phase 1 (`nettoyer`, sans troncature). On **vérifie** que l'artiste et le titre
relus correspondent à ceux de la phase 1 : un écart signale un corpus modifié entre les deux phases
et arrête le carnet.

La justification de phase 1 est conservée sous le nom `justification_phase1` pour la comparer à la
description. Le tableau final donne le nombre d'autoportraits par année.

Si le fichier source n'existe pas encore (campagne de phase 1 en cours), la cellule liste les lots
disponibles : choisir l'un d'eux dans `n_source` pour un essai.

Aucune inférence n'est lancée à cette étape.

In [ ]:
if not source_path.is_file():
    disponibles = sorted(int(p.stem.rsplit('_', 1)[1]) for p in resultats_dir.glob('autoportrait_qwen_annotation_*.csv')
                         if p.stem.rsplit('_', 1)[1].isdigit())
    raise FileNotFoundError(f'{source_path.name} absent (phase 1 non terminée ?). Lots disponibles pour n_source : {disponibles}')
phase1 = pd.read_csv(source_path, dtype={'annee': 'string'}, keep_default_na=False, encoding='utf-8-sig')
colonnes_requises = {'source_row_index', 'artiste', 'titre', 'annee', 'etiquette', 'statut', 'justification'}
if not colonnes_requises <= set(phase1.columns):
    raise ValueError(f'Colonnes manquantes dans {source_path.name} : {sorted(colonnes_requises - set(phase1.columns))}')
if phase1['source_row_index'].duplicated().any():
    raise ValueError('source_row_index en double dans le fichier de phase 1.')
autoportraits = phase1.loc[(phase1['statut'] == 'ok') & (phase1['etiquette'] == 'autoportrait')]
if autoportraits.empty:
    raise ValueError('Aucun autoportrait dans le lot source.')

ids_auto = set(autoportraits['source_row_index'].astype(int))
textes = source.lire_corpus(corpus_path, ids_auto)
chansons = []
for ligne in tqdm(autoportraits.itertuples(index=False), total=len(autoportraits), desc='Paroles', unit='chansons'):
    position = int(ligne.source_row_index)
    texte = textes.get(position)
    if texte is None or (texte['artiste'], texte['titre']) != (ligne.artiste, ligne.titre):
        raise RuntimeError(f'Chanson {position} introuvable ou différente dans corpus.csv : corpus modifié depuis la phase 1 ?')
    paroles = source.nettoyer(texte['paroles'])
    if not paroles:
        raise RuntimeError(f'Paroles vides après nettoyage pour la chanson {position}.')
    chansons.append({'source_row_index': position, 'artiste': ligne.artiste, 'titre': ligne.titre,
                     'annee': ligne.annee, 'mots_nettoyes': len(paroles.split()),
                     'justification_phase1': ligne.justification, 'paroles': paroles})

comptes_phase1 = phase1.loc[phase1['statut'] == 'ok', 'etiquette'].value_counts()
print(f'Phase 1 ({source_path.name}) : {len(phase1)} chansons ; étiquettes valides : {comptes_phase1.to_dict()}')
print(f'Sous-ensemble AUTOPORTRAIT : {len(chansons)} chansons '
      f"({100 * len(chansons) / max(int(comptes_phase1.sum()), 1):.1f} % des annotations valides) ; paroles relues et vérifiées.")
par_annee = pd.Series([c['annee'] or 'NA' for c in chansons]).value_counts().sort_index().rename('autoportraits')
par_annee.index.name = 'annee'
with pd.option_context('display.max_rows', None):
    display(par_annee.to_frame().T)
print('Aucune inférence lancée.')

## 2. Échantillonnage : test ou sous-ensemble complet

- `taille_echantillon = None` : **tous** les autoportraits du lot source (périmètre `autoportraits`).
- `taille_echantillon` entier (par ex. `10`) : **test** tiré au hasard parmi les autoportraits avec la
  graine 42 (périmètre `autoportraits_test`), dans l'ordre du corpus. Le même réglage redonne le même tirage.

Le nom des fichiers de sortie dépend de la taille du lot traité :
`autoportrait2_qwen_description_{n}`. Aucune inférence n'est lancée à cette étape.

**Pondération** : le corpus d'étude sur-représente les années avant 2004 (toutes gardées) par rapport aux
années suivantes (500 par an). Les proportions globales de l'étape 4 décrivent donc le corpus d'étude,
pas toute la base ; le croisement par période permet de les lire séparément.

In [ ]:
if taille_echantillon is None:
    selection = chansons
    perimetre = 'autoportraits'
else:
    if not taille_echantillon <= len(chansons):
        raise ValueError(f'taille_echantillon doit être compris entre 1 et {len(chansons)} (autoportraits disponibles), ou None.')
    indices = sorted(pd.Series(range(len(chansons))).sample(n=taille_echantillon, random_state=graine).tolist())
    selection = [chansons[i] for i in indices]
    perimetre = 'autoportraits_test'
mode_lancement = 'sous-ensemble complet' if taille_echantillon is None else 'test'
print(f'Périmètre : {perimetre} ({mode_lancement}) ; {len(selection)} chansons sur {len(chansons)} autoportraits.')
print(f'Export prévu : autoportrait2_qwen_description_{len(selection)} — aucune inférence lancée à cette étape.')
display(pd.DataFrame(selection)[['source_row_index', 'artiste', 'titre', 'annee', 'mots_nettoyes']].head(10))

## 3. Annotation

### 3.1 Client vLLM et validation de la description

Chaque requête envoie la consigne A en message système et les paroles nettoyées en message
utilisateur (même format qu'en phase 1), avec `temperature=0`, `seed=42`, `max_tokens=8192` et le
raisonnement activé (`chat_template_kwargs.enable_thinking`). Le schéma A est imposé par
`response_format` (`json_schema`, `strict`). **Le moteur de grammaire de vLLM (xgrammar) refuse le
mot-clé `uniqueItems`** (erreur HTTP 400 « Grammar error: Unimplemented keys ») : le serveur reçoit donc
une copie du schéma sans ce mot-clé (`schema_serveur`), et l'absence de doublons est contrôlée ici.
Le fichier `schema_A_description.json` n'est pas modifié.

La réponse est revalidée côté client à partir du schéma lui-même :

- génération terminée (`finish_reason == "stop"`) et raisonnement non vide ;
- exactement les six clés du schéma ;
- `justification` texte non vide ;
- `registre`, `temporalite`, `regard_des_autres` dans leurs valeurs autorisées ;
- `aspects` : 1 à 4 valeurs autorisées, sans doublon, **ordre conservé** (du plus au moins important) ;
- `demarcation` : valeurs autorisées sans doublon, liste vide acceptée.

Les serveurs sont répartis par une file : chaque worker prend un serveur libre, ce qui équilibre la
charge entre les deux GPU. Les erreurs HTTP transitoires (408, 429, 5xx) et les réponses invalides
sont retentées.

In [ ]:
ouvreur = urllib.request.build_opener(urllib.request.ProxyHandler({}))

def appeler_api(route, corps=None, *, serveur):
    requete = urllib.request.Request(
        serveur + route,
        data=None if corps is None else json.dumps(corps).encode('utf-8'),
        headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {cle_api}'},
    )
    with ouvreur.open(requete, timeout=timeout_secondes) as reponse:
        return json.loads(reponse.read().decode('utf-8'))

def verifier_endpoint(serveur):
    try:
        reponse = appeler_api('/models', serveur=serveur)
    except urllib.error.HTTPError as exc:
        raise RuntimeError(f'Serveur vLLM : HTTP {exc.code} sur {serveur}/models. Vérifier la configuration et la clé API.') from exc
    except (urllib.error.URLError, OSError) as exc:
        raise RuntimeError(
            f'Serveur Qwen injoignable sur {serveur}. Démarrer vLLM avec --reasoning-parser qwen3 '
            'sur un GPU libre, attendre sa disponibilité, puis relancer la cellule. '
            'Le notebook ne démarre pas le serveur automatiquement.') from exc
    modeles = [item['id'] for item in reponse['data']]
    if modele not in modeles:
        raise RuntimeError(f'{modele} non servi sur {serveur}. Modèles disponibles : {modeles}')
    print('Serveur disponible :', serveur, '|', modele)

def verifier_serveur():
    for serveur in base_urls:
        verifier_endpoint(serveur)

def valider_description(objet, raisonnement):
    if enable_thinking and (not isinstance(raisonnement, str) or not raisonnement.strip()):
        raise ValueError('Raisonnement absent : vérifier enable_thinking et --reasoning-parser qwen3 côté serveur.')
    if not isinstance(objet, dict) or set(objet) != set(champs):
        raise ValueError(f'Clés exactes attendues : {champs}.')
    if not isinstance(objet['justification'], str) or not objet['justification'].strip():
        raise ValueError('Justification vide ou non textuelle.')
    description = {'justification': objet['justification'].strip()}
    for champ in champs_simples:
        if objet[champ] not in modalites[champ]:
            raise ValueError(f'{champ} inconnu : {objet[champ]!r}')
        description[champ] = objet[champ]
    for champ in champs_listes:
        valeurs = objet[champ]
        bornes = proprietes[champ].get('minItems', 0), proprietes[champ].get('maxItems', len(modalites[champ]))
        if not isinstance(valeurs, list) or not bornes[0] <= len(valeurs) <= bornes[1]:
            raise ValueError(f'{champ} : liste de {bornes[0]} à {bornes[1]} valeurs attendue.')
        if len(set(valeurs)) != len(valeurs) or any(v not in modalites[champ] for v in valeurs):
            raise ValueError(f'{champ} : valeurs inconnues ou répétées : {valeurs!r}')
        description[champ] = list(valeurs)
    description['raisonnement'] = raisonnement.strip() if isinstance(raisonnement, str) else ''
    return description

def valider_reponse(reponse):
    choix = reponse['choices'][0]
    if choix.get('finish_reason') != 'stop':
        raise ValueError(f"Génération incomplète : {choix.get('finish_reason')!r}")
    message = choix['message']
    if not isinstance(message['content'], str):
        raise ValueError('Contenu absent ou non textuel.')
    return valider_description(json.loads(message['content']),
                               message.get('reasoning') or message.get('reasoning_content'))

def annoter_chanson(chanson):
    serveur = serveurs_disponibles.get()
    try:
        return {**annoter_sur_serveur(chanson, serveur), 'serveur': serveur}
    finally:
        serveurs_disponibles.put(serveur)

def description_vide():
    return {'justification': '', **{c: '' for c in champs_simples}, **{c: [] for c in champs_listes}, 'raisonnement': ''}

def annoter_sur_serveur(chanson, serveur):
    debut = time.perf_counter()
    corps = {
        'model': modele,
        'messages': source.messages(consigne, chanson['paroles'], False),
        'temperature': 0, 'seed': graine, 'max_tokens': max_tokens,
        'chat_template_kwargs': {'enable_thinking': enable_thinking},
        **contrainte,
    }
    for essai in range(1, max_essais + 1):
        try:
            reponse = appeler_api('/chat/completions', corps, serveur=serveur)
            return {**valider_reponse(reponse), 'statut': 'ok', 'erreur': '', 'essais': essai,
                    'secondes': time.perf_counter() - debut}
        except urllib.error.HTTPError as exc:
            erreur = f'HTTP {exc.code} : {exc.read().decode("utf-8", "replace")[:1000]}'
            if exc.code not in {408, 429, 500, 502, 503, 504}:
                break
        except (OSError, ValueError, KeyError, IndexError, TypeError) as exc:
            erreur = f'{type(exc).__name__} : {exc}'
        if essai < max_essais:
            time.sleep(2 * essai)
    return {**description_vide(), 'statut': 'erreur', 'erreur': erreur, 'essais': essai,
            'secondes': time.perf_counter() - debut}

### 3.2 Pipeline : journal, reprise et exports

- **Empreinte** : condensat de la configuration (modèle, serveurs, consigne A, schéma A,
  `annoter.py`, paramètres de génération, lot source et sélection). Elle nomme le journal
  `autoportrait2_qwen32_reason_{perimetre}_n{n}_{empreinte}.jsonl` : changer la consigne, le schéma
  ou la sélection ouvre un nouveau journal au lieu de mélanger des résultats incompatibles.
- **Reprise** : chaque résultat est écrit et synchronisé sur disque dès son arrivée. À la relance, les
  réussites du journal sont revalidées et reprises ; seules les chansons absentes ou en erreur sont
  renvoyées au modèle. Une interruption ne perd donc rien.
- **Verrou** : `autoportrait2_qwen_description_{n}.lock` empêche deux exécutions simultanées d'écrire
  les mêmes fichiers.
- **Exports** :
  - CSV : listes jointes par `|`, `aspect_principal` = premier aspect cité, indicatrices 0/1
    `aspect_*` et `demarcation_*` ;
  - JSON : listes conservées ;
  - CSV des erreurs.
- **Performances** : une ligne `date,modele,batch,concurrent,n,chansons/s` est ajoutée à
  `data/performance/perfGPU_autoportrait2.csv` à chaque essai, de sorte que les mesures
  s'accumulent. `chansons/s` est vide si tout a été repris du journal.

In [ ]:
def empreinte(selection):
    configuration = {
        'version_pipeline': 1, 'tache': 'consigne_A_description', 'modele': modele, 'urls': base_urls,
        'consigne': consigne, 'schema': schema, 'contrainte': contrainte,
        'annoter_sha256': hashlib.sha256((dossier / 'annoter.py').read_bytes()).hexdigest(),
        'temperature': 0, 'seed': graine, 'max_tokens': max_tokens, 'enable_thinking': enable_thinking,
        'source': source_path.name, 'selection': selection,
    }
    return hashlib.sha256(json.dumps(configuration, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def tableau_export(resultats):
    tableau = pd.DataFrame(resultats)
    for champ in champs_listes:
        for valeur in modalites[champ]:
            tableau[f'{champ.removesuffix("s")}_{valeur}'] = tableau[champ].map(lambda liste, v=valeur: int(v in liste))
    tableau.insert(tableau.columns.get_loc('aspects') + 1, 'aspect_principal',
                   tableau['aspects'].map(lambda liste: liste[0] if liste else ''))
    for champ in champs_listes:
        tableau[champ] = tableau[champ].map('|'.join)
    return tableau

def executer_description(selection, perimetre):
    if not selection:
        raise ValueError('Sélection vide.')
    resultats_dir.mkdir(parents=True, exist_ok=True)
    performance_dir.mkdir(parents=True, exist_ok=True)
    signature = empreinte(selection)
    mode = 'reason' if enable_thinking else 'sans_reason'
    prefixe = f'autoportrait2_qwen32_{mode}_{perimetre}_n{len(selection)}_{signature[:12]}'
    prefixe_sortie = f'autoportrait2_qwen_description_{len(selection)}'
    journal = resultats_dir / f'{prefixe}.jsonl'
    ids = [c['source_row_index'] for c in selection]
    ids_selection = set(ids)
    if len(ids) != len(ids_selection):
        raise ValueError('Sélection avec doublons.')
    debut = time.perf_counter()
    with (resultats_dir / f'{prefixe_sortie}.lock').open('a') as verrou:
        try:
            fcntl.flock(verrou, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError('Un lot de même taille écrit déjà ces fichiers de sortie.') from exc
        resultats = {}
        if journal.exists():
            with journal.open(encoding='utf-8') as fichier:
                for numero, ligne in enumerate(fichier, 1):
                    try:
                        resultat = json.loads(ligne)
                    except json.JSONDecodeError as exc:
                        raise ValueError(f'Journal endommagé : {journal}, ligne {numero}. Réparer avant reprise.') from exc
                    if resultat['empreinte'] != signature or resultat['source_row_index'] not in ids_selection:
                        raise ValueError('Journal incompatible avec la sélection courante.')
                    if resultat['statut'] not in {'ok', 'erreur'}:
                        raise ValueError('Statut invalide dans le journal.')
                    if resultat['statut'] == 'ok':
                        valider_description({c: resultat[c] for c in champs}, resultat.get('raisonnement'))
                    resultats[resultat['source_row_index']] = resultat
        restantes = [c for c in selection if resultats.get(c['source_row_index'], {}).get('statut') != 'ok']
        reprises = len(selection) - len(restantes)
        print(f'{perimetre} : {reprises} réussites reprises ; {len(restantes)} chansons à traiter.')
        if restantes:
            verifier_serveur()
        traitees = erreurs = 0
        with journal.open('a', encoding='utf-8') as fichier:
            with ThreadPoolExecutor(max_workers=concurrence) as executeur:
                taches = {}
                try:
                    with tqdm(total=len(selection), initial=reprises, desc=f'Description {perimetre}',
                              unit='chansons', mininterval=0.5) as barre:
                        for debut_lot in range(0, len(restantes), taille_lot):
                            lot = restantes[debut_lot:debut_lot + taille_lot]
                            taches = {executeur.submit(annoter_chanson, c): c for c in lot}
                            for tache in as_completed(taches):
                                chanson = taches[tache]
                                resultat = {**{k: v for k, v in chanson.items() if k != 'paroles'},
                                            **tache.result(), 'modele': modele, 'empreinte': signature,
                                            'date_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
                                fichier.write(json.dumps(resultat, ensure_ascii=False) + '\n')
                                fichier.flush()
                                os.fsync(fichier.fileno())
                                resultats[chanson['source_row_index']] = resultat
                                traitees += 1
                                erreurs += resultat['statut'] == 'erreur'
                                barre.update(1)
                                barre.set_postfix(erreurs=erreurs,
                                    chansons_s=f'{traitees / max(time.perf_counter() - debut, 1e-9):.2f}', refresh=False)
                except BaseException:
                    for tache in taches:
                        tache.cancel()
                    raise
        ordonnes = [resultats[position] for position in ids]
        if len(ordonnes) != len(selection):
            raise RuntimeError('Effectif final incohérent.')
        tableau = tableau_export(ordonnes)
        tableau.to_csv(resultats_dir / f'{prefixe_sortie}.csv', index=False, encoding='utf-8-sig')
        (resultats_dir / f'{prefixe_sortie}.json').write_text(
            json.dumps(ordonnes, ensure_ascii=False, indent=2), encoding='utf-8')
        tableau.loc[tableau['statut'] == 'erreur'].to_csv(
            resultats_dir / f'{prefixe_sortie}_erreurs.csv', index=False, encoding='utf-8-sig')
        secondes = time.perf_counter() - debut
        metriques = {'perimetre': perimetre, 'source': source_path.name, 'modele': modele, 'empreinte': signature,
                     'selection': len(selection), 'reprises': reprises, 'traitees_session': traitees,
                     'reussites': int((tableau['statut'] == 'ok').sum()),
                     'erreurs': int((tableau['statut'] == 'erreur').sum()),
                     'secondes_session_client': secondes, 'chansons_s_session': traitees / secondes,
                     'date_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                     'type_execution': 'reprise_cache' if not traitees else ('mixte' if reprises else 'calcul'),
                     'nombre_serveurs': len(base_urls), 'concurrence': concurrence,
                     'concurrence_par_serveur': concurrence_par_serveur, 'taille_lot': taille_lot,
                     'max_tokens': max_tokens, 'enable_thinking': enable_thinking,
                     'raisonnements_non_vides': int(tableau['raisonnement'].str.strip().ne('').sum())}
        performance_essai = pd.DataFrame([{
            'date': metriques['date_utc'], 'modele': modele, 'batch': taille_lot,
            'concurrent': concurrence, 'n': len(selection),
            'chansons/s': traitees / secondes if traitees else None,
        }])
        with cumul_path.open('a+', encoding='utf-8', newline='') as cumul:
            fcntl.flock(cumul, fcntl.LOCK_EX)
            cumul.seek(0, os.SEEK_END)
            vide = cumul.tell() == 0
            if not vide:
                cumul.seek(0)
                if list(pd.read_csv(cumul, nrows=0).columns) != list(performance_essai.columns):
                    raise ValueError(f'Colonnes incompatibles dans {cumul_path}. Corriger le fichier avant de relancer.')
            cumul.seek(0, os.SEEK_END)
            performance_essai.to_csv(cumul, index=False, header=vide)
            cumul.flush()
            os.fsync(cumul.fileno())
    display(pd.DataFrame([metriques]))
    if metriques['erreurs']:
        print('ATTENTION : erreurs techniques présentes ; voir le CSV erreurs et relancer pour les retenter.')
    print('Descriptions :', resultats_dir / f'{prefixe_sortie}.csv')
    print('Performances cumulées :', cumul_path)
    return ordonnes, tableau

### 3.3 Lancement

**Avant de lancer : vérifier les GPU (`nvidia-smi`) et que les deux serveurs répondent sur `/v1/models`.**
La cellule décrit la sélection de l'étape 2 ; relancer la même cellule reprend un lot interrompu.
Elle s'arrête en erreur si des chansons restent en échec, après avoir écrit tous les fichiers.

In [ ]:
descriptions, resultats = executer_description(selection, perimetre)
display(resultats[['source_row_index', 'artiste', 'titre', 'annee', 'registre', 'aspects',
                   'temporalite', 'demarcation', 'regard_des_autres', 'justification', 'erreur']].head(20))
if not resultats['statut'].eq('ok').all():
    raise RuntimeError('Lot incomplet : consulter le fichier erreurs et relancer pour reprendre.')

## 4. Synthèse

Calculée sur les descriptions valides du lot :

- `registre`, `temporalite`, `regard_des_autres` : effectif et pourcentage de chaque valeur ;
- `aspects` : part des chansons qui citent chaque aspect (**total > 100 %**, jusqu'à 4 aspects par
  chanson) et part où il est l'**aspect principal** (premier cité) ;
- `demarcation` : part des chansons qui s'opposent à chaque figure (total > 100 % possible), plus
  `aucune` pour les listes vides ;
- croisement **registre × période** (avant 2004 / à partir de 2004), en pourcentages par ligne, pour
  lire séparément les deux strates du corpus d'étude.

Export : `autoportrait2_qwen_description_{n}_distributions.csv`, avec les colonnes
`variable, modalite, effectif, pourcentage, principal, pourcentage_principal` (les deux dernières pour
`aspects` seulement).

In [ ]:
valides = [d for d in descriptions if d['statut'] == 'ok']
if not valides:
    raise RuntimeError('Aucune description valide à synthétiser.')
lignes = []
for champ in champs_simples:
    comptes = pd.Series([d[champ] for d in valides]).value_counts()
    lignes += [{'variable': champ, 'modalite': m, 'effectif': int(comptes.get(m, 0))} for m in modalites[champ]]
for champ in champs_listes:
    principaux = pd.Series([d[champ][0] for d in valides if d[champ]]).value_counts()
    for m in modalites[champ]:
        ligne = {'variable': champ, 'modalite': m, 'effectif': sum(m in d[champ] for d in valides)}
        if champ == 'aspects':
            ligne['principal'] = int(principaux.get(m, 0))
        lignes.append(ligne)
    if proprietes[champ].get('minItems', 0) == 0:
        lignes.append({'variable': champ, 'modalite': 'aucune', 'effectif': sum(not d[champ] for d in valides)})
distributions = pd.DataFrame(lignes)
distributions['pourcentage'] = (100 * distributions['effectif'] / len(valides)).round(1)
distributions['pourcentage_principal'] = (100 * distributions['principal'] / len(valides)).round(1)
distributions = distributions[['variable', 'modalite', 'effectif', 'pourcentage', 'principal', 'pourcentage_principal']]
distributions['principal'] = distributions['principal'].astype('Int64')
distributions_path = resultats_dir / f'autoportrait2_qwen_description_{len(descriptions)}_distributions.csv'
distributions.to_csv(distributions_path, index=False, encoding='utf-8')
print(f'{len(valides)} descriptions valides sur {len(descriptions)}.')
for variable, bloc in distributions.groupby('variable', sort=False):
    print('\n' + variable)
    colonnes = ['modalite', 'effectif', 'pourcentage'] + (['principal', 'pourcentage_principal'] if variable == 'aspects' else [])
    display(bloc[colonnes].sort_values('effectif', ascending=False).reset_index(drop=True))

def periode(annee, seuil=2004):
    texte = str(annee).strip()
    return 'NA' if not texte.isdigit() else (f'avant {seuil}' if int(texte) < seuil else f'{seuil} et après')

periodes = pd.Series([periode(d['annee']) for d in valides])
croisement = pd.crosstab(periodes.rename('periode'), pd.Series([d['registre'] for d in valides], name='registre'),
                         normalize='index').mul(100).round(1)
croisement['chansons'] = periodes.value_counts()
print('\nRegistre × période (% par ligne)')
display(croisement)
print('Distributions :', distributions_path)